# Kuramoto-Sivashinsky equation

$$u_t + \alpha\,u\,u_x + \beta\,u_{xx} + \gamma\,u_{xxxx} = 0,\qquad x\in[0,2\pi],\ t\in(0,1)$$

with periodic boundaries and $u(x,0)=\cos(x)\,(1+\sin(x))$.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

import pinn
from pinn import operators as op, sampling

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision

In [ ]:
X, T = 0, 1


class KSProblem(pinn.Problem):
    """u_t + alpha u u_x + beta u_xx + gamma u_xxxx = 0, periodic on [0, 2pi]."""

    x_min, x_max, t_max = 0.0, 2.0 * jnp.pi, 1.0
    alpha, beta, gamma = 100.0 / 16.0, 100.0 / 16.0**2, 100.0 / 16.0**4
    problem_name = "KS"
    ref_path = pinn.reference_path("ks")

    def __init__(self, *, n_pde, n_ic, n_bc=0):
        self.n_pde, self.n_ic, self.n_bc = n_pde, n_ic, n_bc

    def residual_fns(self):
        return {"pde": self.pde_residual, "ic": self.ic_residual}

    def pde_residual(self, model, coords):
        u = lambda c: model(c)[0]
        r = (op.grad(u, coords, T)
             + self.alpha * u(coords) * op.grad(u, coords, X)
             + self.beta * op.grad2(u, coords, X)
             + self.gamma * op.grad_n(u, coords, X, 4))
        return jnp.array([r])

    def ic_residual(self, model, coords):
        return jnp.array([model(coords[:2])[0] - coords[2]])

    def samplers(self):
        return {"pde": sampling.interior(self.x_min, self.x_max),
                "ic":  sampling.initial_line(self.x_min, self.x_max)}

    def analytical_ic(self, x):
        return jnp.cos(x) * (1.0 + jnp.sin(x))

In [ ]:
# Slab edges clustered towards t = 1, where the dynamics are sharpest.
slab_boundaries = jnp.hstack([0.0, 1.0 - ((jnp.arange(8) / 8)**2)[::-1]])

cfg = pinn.RunConfig(
    slab_boundaries=list(slab_boundaries),
    network=lambda key: pinn.MLP(
        key, KSProblem, hidden_dims=(70, 70, 70, 70),
        periodic_bc=True, n_inputs=2, n_outputs=1,
    ),
    n_pde=2**15, n_ic=2**14,
    residual_sketch=5000, parameter_sketch=5000,
    batch_size=2**12, probe_batch_size=2**8,
    pde_weight=1e-3,
)

In [ ]:
pinn.precompile64(KSProblem, cfg)

In [ ]:
results = pinn.train64(KSProblem, cfg)

## Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.load_reference("ks")
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()